# Criador de Prontuário — versão Python (equivalente ao fluxo n8n)

Este notebook reproduz, em Python, o fluxo n8n **"Criador de prontuário"**:

1. **Agente Planejador** (LLM) — planeja N prontuários (conformes / parcialmente conformes)
2. **Divide planejamento** — quebra a resposta em 1 plano por prontuário
3. **Preenche dados pré-definidos** — gera campos determinísticos (número de prontuário, atendimento, datas, etc.) e monta os registros (Anamnese/Evolução por categoria e dia)
4. **Prepara body request** — distribui as falhas planejadas por registro e monta o prompt do Agente Gerador
5. **Agente Gerador** (LLM) — gera o texto clínico de cada registro
6. **Monta json final** — faz o parse da resposta da LLM e mescla no registro
7. **Prepara json e csv** — agrega as falhas por atendimento e monta o gabarito (CSV) + o `.txt` com os prontuários completos
8. **Salva no arquivo local** — grava (em modo append) os arquivos finais

> Ajuste as variáveis da célula **Configuração** antes de rodar.


## Configuração

In [61]:

import os
import re
import json
import random
import string
import requests
from datetime import datetime, timedelta

# ── Credenciais via .env (NUNCA cole a chave direto no código) ──────────
# Crie um arquivo ".env" na mesma pasta deste notebook com a linha:
#   NVIDIA_API_KEY=sua_chave_aqui
# e adicione ".env" ao seu .gitignore, para nunca versionar a chave.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    print("Aviso: python-dotenv não instalado — rode 'pip install python-dotenv' "
          "ou defina NVIDIA_API_KEY manualmente antes de continuar.")

NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY")
NVIDIA_API_KEY_KIMI_K3 = os.environ.get("NVIDIA_API_KEY_KIMI_K3")
OPENROUTER_KEY = os.environ.get("OPENROUTER_API_KEY")
if not NVIDIA_API_KEY:
    raise RuntimeError(
        "NVIDIA_API_KEY não encontrada. Defina no arquivo .env ou via "
        "'export NVIDIA_API_KEY=...' antes de abrir o Jupyter."
    )

if not NVIDIA_API_KEY_KIMI_K3:
    raise RuntimeError(
        "NVIDIA_API_KEY_KIMI_K3 não encontrada. Defina no arquivo .env ou via "
        "'export NVIDIA_API_KEY_KIMI_K3=...' antes de abrir o Jupyter."
    )

NVIDIA_URL = "https://integrate.api.nvidia.com/v1/chat/completions"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_KEY}",
    "Accept": "application/json",
    "Content-Type": "application/json",
}

# ── Modelos usados em cada etapa (iguais ao fluxo n8n) ──────────────────
# MODELO_PLANEJADOR = "moonshotai/kimi-k3"
# MODELO_GERADOR    = "deepseek-ai/deepseek-v4-flash-0731"
MODELO_PLANEJADOR = "deepseek/deepseek-v4-flash-0731:free"
MODELO_GERADOR    = "deepseek/deepseek-v4-flash-0731:free"

# Fallback do Agente Gerador: só é usado quando a API sinaliza servidor
# sobrecarregado (429/5xx) -- nesse caso a MESMA chamada é refeita na hora
# com este modelo, em vez de esperar o backoff normal com o flash.
# Qualquer outro tipo de erro (timeout, conexão, resposta vazia) continua
# esperando o backoff normalmente, sem trocar de modelo.
MODELO_GERADOR_FALLBACK = MODELO_PLANEJADOR

# ── Parâmetros de geração ────────────────────────────────────────────────
QUANTIDADE_PRONTUARIOS = 3          # quantos prontuários pedir ao planejador
TIMEOUT_REQUEST = 9000              # segundos

# ── Caminhos de saída — AJUSTE para uma pasta sua ────────────────────────
OUTPUT_DIR = "./"
CSV_PATH = os.path.join(OUTPUT_DIR, "gabarito.csv")
JSON_PATH = os.path.join(OUTPUT_DIR, "prontuarios.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)


## Helpers (equivalente aos helpers do node "Preenche dados pré-definidos")

In [62]:

def random_int(a, b):
    return random.randint(a, b)

def format_num(n, digits):
    return str(n).zfill(digits)

def gerar_prontuario():
    return f"{random_int(10,99)}.{format_num(random_int(0,999),3)}.{format_num(random_int(0,999),3)}"

def gerar_atendimento():
    return f"{random_int(1,9)}.{format_num(random_int(0,999),3)}.{format_num(random_int(0,999),3)}"

def gerar_codigo_sus():
    return f"{format_num(random_int(0,999),3)}.{format_num(random_int(0,999),3)}.{format_num(random_int(0,999),3)}"

def _parse_data_hora(data_str):
    """'DD/MM/AAAA, HH:MM' -> (datetime, 'HH:MM')"""
    date_part, time_part = data_str.split(", ")
    dia, mes, ano = map(int, date_part.split("/"))
    return datetime(ano, mes, dia), time_part

def add_days_to_date_string(data_str, days):
    if not data_str:
        return ""
    dt, time_part = _parse_data_hora(data_str)
    dt = dt + timedelta(days=days)
    return f"{dt.day:02d}/{dt.month:02d}/{dt.year}, {time_part}"

def calcular_nascimento(idade_aproximada, data_internacao):
    dt, _ = _parse_data_hora(data_internacao)
    ano_nasc = dt.year - idade_aproximada
    mes_nasc = random_int(1, 12)
    dia_max = [31,28,31,30,31,30,31,31,30,31,30,31][mes_nasc-1]
    dia_nasc = random_int(1, dia_max)
    return f"{dia_nasc}/{mes_nasc}/{ano_nasc}"


## Retry com backoff exponencial

Antes de fazer qualquer chamada à API: se der erro (rate limit `429`, erro de
servidor `5xx`, ou erro de conexão), espera um tempo que **dobra a cada
tentativa** antes de tentar de novo. Depois de `MAX_TENTATIVAS` falhas
seguidas, desiste daquele item específico — ele **não é salvo** no resultado
final, mas o restante do lote continua normalmente.

In [63]:

import time

MAX_TENTATIVAS = 5
BACKOFF_BASE_SEGUNDOS = 5  # tempo de espera antes da 1ª nova tentativa; dobra a cada erro


def _descricao_breve_erro(resp):
    """Extrai uma descrição curta do erro a partir do corpo da resposta,
    tentando o formato {"error": {"message": "..."}} e caindo para o texto
    bruto (truncado) se não for JSON ou não tiver esse formato."""
    try:
        corpo = resp.json()
        msg = corpo.get("error", {}).get("message") if isinstance(corpo, dict) else None
        if msg:
            return msg[:200]
    except Exception:
        pass
    texto = (resp.text or resp.reason or "").strip()
    return texto[:200] if texto else "sem descrição"


def chamar_com_retry(url, headers, body, timeout=TIMEOUT_REQUEST, max_tentativas=MAX_TENTATIVAS,
                      modelo_fallback=None):
    """Faz o POST com retry e backoff exponencial.
    - 429 (rate limit) ou 5xx (erro de servidor = servidor sobrecarregado): se
      `modelo_fallback` foi informado e ainda não caímos nele, a MESMA chamada
      é refeita IMEDIATAMENTE (sem esperar backoff) trocando o "model" do body
      para o fallback. Só depois desse fallback (ou se não houver fallback
      configurado) é que passa a esperar e tentar de novo com backoff normal.
    - Erro de conexão/timeout, ou 200 OK sem conteúdo: sempre espera o backoff
      normal e tenta de novo com o MESMO modelo (não troca de modelo nesses
      casos).
    - Qualquer outro erro HTTP (4xx que não seja 429): falha imediatamente,
      sem retry, porque é um erro de requisição malformada — tentar de novo
      não vai resolver.
    Retorna o JSON da resposta em caso de sucesso, ou None se esgotar as
    tentativas (o chamador decide o que fazer — normalmente descartar o item)."""
    espera = BACKOFF_BASE_SEGUNDOS
    body_atual = dict(body)
    ja_caiu_no_fallback = False

    for tentativa in range(1, max_tentativas + 1):
        try:
            resp = requests.post(url, headers=headers, json=body_atual, timeout=timeout)

            if resp.status_code == 429 or resp.status_code >= 500:
                descricao = _descricao_breve_erro(resp)

                pode_cair_no_fallback = (
                    modelo_fallback
                    and not ja_caiu_no_fallback
                    and body_atual.get("model") != modelo_fallback
                )
                if pode_cair_no_fallback:
                    print(f"Erro {resp.status_code} - {descricao} (servidor sobrecarregado). "
                          f"Caindo para o modelo de fallback '{modelo_fallback}' agora, sem esperar.")
                    body_atual["model"] = modelo_fallback
                    ja_caiu_no_fallback = True
                    continue  # tenta de novo já com o modelo novo, sem sleep

                print(f"Erro {resp.status_code} - {descricao}, fazendo a requisição de novo "
                      f"em {espera} segundos (tentativa {tentativa}/{max_tentativas})")
                if tentativa < max_tentativas:
                    time.sleep(espera)
                    espera *= 2
                continue

            resp.raise_for_status()  # outros erros 4xx: levanta na hora, sem retry
            resultado = resp.json()

            # 200 OK mas sem "choices"/"content" -- resposta malformada da API,
            # mesmo comportamento de retry de um erro de servidor (sem trocar modelo)
            tem_conteudo = bool(resultado.get("choices")) or bool(resultado.get("content"))
            if not tem_conteudo:
                print(f"Erro 200-vazio - resposta sem 'choices'/'content', fazendo a "
                      f"requisição de novo em {espera} segundos (tentativa {tentativa}/{max_tentativas})")
                if tentativa < max_tentativas:
                    time.sleep(espera)
                    espera *= 2
                continue

            return resultado

        except requests.exceptions.HTTPError:
            raise  # erro 4xx que não seja 429 — não adianta tentar de novo

        except requests.exceptions.RequestException as e:
            print(f"Erro de conexão/timeout - {e}, fazendo a requisição de novo "
                  f"em {espera} segundos (tentativa {tentativa}/{max_tentativas})")
            if tentativa < max_tentativas:
                time.sleep(espera)
                espera *= 2

    print(f"-> desistindo após {max_tentativas} tentativas. Este item NÃO será salvo.")
    return None


## Etapa 1 — Agente Planejador

In [ ]:

SYSTEM_PROMPT_PLANEJADOR = r"""# IDENTIDADE
Você é um agente PLANEJADOR de prontuários hospitalares fictícios para treinamento de IA em auditoria. Sua única função é planejar cenários — NÃO gera os prontuários.

# SAÍDA
Retorne APENAS o JSON abaixo, sem explicações, sem markdown:
{
  "total_prontuarios": N,
  "prontuarios": [{
    "id_planejamento": "PLAN-001",
    "classificacao": "CONFORME | PARCIALMENTE CONFORME",
    "especialidade": "",
    "diagnostico_principal": "",
    "cid_principal": "",
    "procedimento": "",
    "sexo": "M | F",
    "idade_aproximada": N,
    "duracao_internacao_dias": N,
    "data_internacao": "DD/MM/AAAA, HH:MM",
    "data_saida": "DD/MM/AAAA, HH:MM",
    "unidade_funcional": "",
    "utiliza_o2": "Sim | Não",
    "antibiotico_profilatico": "Sim | Não",
    "categorias_envolvidas": ["MEDICINA", "ENFERMAGEM"],
    "tem_cirurgia": true,
    "data_inicio_cirurgia": "DD/MM/AAAA, HH:MM",
    "data_fim_cirurgia": "DD/MM/AAAA, HH:MM",
    "falhas_planejadas": [{
      "secao": "",
      "campo": "",
      "resultado": "ausente | incompleta",
      "categoria_profissional": "",
      "descricao_falha": "",
      "dias_ausentes": []
    }],
    "observacoes_clinicas": ""
  }]
}

# CAMPOS VÁLIDOS POR SEÇÃO
- secao_a: prontuario, data_nascimento, idade, especialidade_internacao, periodo_internacao, diagnostico_internacao, especialidade_cirurgia, unidade_funcional
- secao_b_anamnese (MEDICINA/Anamnese): hda, hd_cid, ap_app, af, exame_fisico, cd, criacao_anamnese
- secao_b_evolucao (MEDICINA/Evolução): hd_cid, exame_fisico, procedimentos_condutas_queixas, frequencia_diaria
- secao_c (cirúrgico, só se tem_cirurgia=true): tem_cirurgia, especialidade, unidade_funcional, inicio, fim, diagnostico_cid, descricao_procedimento, descricao_tecnica, uso_opme
- secao_d_anamnese (ENFERMAGEM/Anamnese): motivo_internacao, ap_app, af, exame_fisico, escala_braden, escala_morse, cd, criacao_anamnese, curativo
- secao_d_evolucao (ENFERMAGEM/Evolução): motivo_internacao, exame_fisico, condutas, escala_braden, escala_morse, criacao_evolucao, curativo
- secao_e (outras categorias): tem_outras_categorias, categoria, descricao

# REGRAS
R1. Nunca repetir cenário clínico no lote — diversificar diagnósticos, especialidades, perfis
R1b. id_planejamento deve ser único e sequencial dentro do lote (PLAN-001, PLAN-002, PLAN-003...)
R2. A QUANTIDADE de falhas_planejadas de cada prontuário vem definida na mensagem do usuário (uma quantidade por prontuário, na ordem) -- NÃO decida essa quantidade sozinho. Se a quantidade informada for 0: classificacao=CONFORME e falhas_planejadas=[].
R3. Se a quantidade informada for maior que 0: classificacao=PARCIALMENTE CONFORME e falhas_planejadas deve conter EXATAMENTE essa quantidade de itens (nem a mais, nem a menos).
R4. Diversificar secao/campo das falhas entre prontuários. Máximo 2 com dias_ausentes parciais
R5. Sempre incluir MEDICINA e ENFERMAGEM em categorias_envolvidas
R6. tem_cirurgia=true → secao_c implícita no plano
R7. CIDs compatíveis com diagnóstico; falhas coerentes com perfil clínico
R8. Não incluir Prontuário/Atendimento/DataNascimento/CodigoSUS (gerados por código)
R9. Internações sem cirurgia: máximo 5 dias. Com cirurgia: máximo 4 dias
R10. data_inicio/fim_cirurgia dentro do período de internação, nunca no 1º dia
R11. antibiotico_profilatico="Sim" apenas para cirurgias
R12. dias_ausentes só quando campo=frequencia_diaria ou criacao_evolucao com resultado=incompleta
R13. Usar exatamente os nomes de secao/campo listados acima
R14. Priorizar falhas de campo (af, hda, exame_fisico, diagnostico_cid) sobre dias_ausentes
R15. exame_fisico incompleto: especificar na descricao_falha quais sistemas faltam
  - Médico: GERAL, ACV, AR, ABD, EXT
  - Enfermagem: SNC, Pele/Mucosas, Resp, Cardiovascular, GI, GU, Músculo-esquelético, Escalas
R16. Quando a quantidade exigida for alta (5-8), diversifique BASTANTE entre seções/campos diferentes (e repita secao/campo com dias_ausentes diferentes se precisar) para não ficar artificial -- mas o número de itens em falhas_planejadas tem que bater exatamente com o pedido.

# FALHAS MAIS COMUNS (usar para diversificar)
1. secao_d_anamnese / af / ausente — AF omitido na enfermagem
2. secao_c / diagnostico_cid / ausente — CID cirúrgico em branco
3. secao_b_anamnese / exame_fisico / incompleta — faltam aparelhos no exame médico
4. secao_b_anamnese / af / ausente — AF omitido na anamnese médica
5. secao_b_anamnese / criacao_anamnese / incompleta — anamnese feita >12h após internação
6. secao_d_evolucao / curativo / incompleta — curativo citado sem o tipo (simples/especial/grau II) e/ou sem tamanho, exsudato ou necrose
7. secao_b_evolucao / hd_cid / ausente — evolução sem hipótese diagnóstica
8. secao_b_evolucao / procedimentos_condutas_queixas / ausente — evolução sem conduta

❌ Não usar resultado="presente" ou "não se aplica" em falhas_planejadas
❌ Não concentrar falhas em dias_ausentes"""


def chamar_agente_planejador(quantidade):
    # Sorteia, EM CÓDIGO (não deixa a LLM decidir), quantas falhas_planejadas
    # cada prontuário do lote vai ter -- entre 0 e 8. Antes a LLM tendia a
    # sempre gerar 2-3 falhas por conta da R16; agora a quantidade é fixada
    # aqui e a LLM só precisa cumprir exatamente o que foi sorteado.
    quantidades_falhas = [random_int(0, 8) for _ in range(quantidade)]
    lista_quantidades = "; ".join(
        f"prontuário {i + 1} = {q} falha(s)" for i, q in enumerate(quantidades_falhas)
    )

    user_content = (
        f"Gere {quantidade} prontuários com diversidade. Inclua pelo 90% de pacientes "
        "com cirurgia. Varie especialidades, diagnósticos, perfis e tipos de falha. Inclua bastante falhas de tipo incompleto "
        "Aplique as regras R1-R21 do planejamento para garantir que nenhum cenário se repita "
        "e que as falhas sejam distribuídas entre os diferentes campos e seções. "
        "é MUITO IMPORTANTE que não tenha mais de 10 registros para um prontuário.\n\n"
        "QUANTIDADE EXATA DE FALHAS POR PRONTUÁRIO (na mesma ordem em que forem gerados -- "
        f"siga à risca, não decida a quantidade sozinho): {lista_quantidades}."
    )

    body = {
        "model": MODELO_PLANEJADOR,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT_PLANEJADOR},
            {"role": "user", "content": user_content},
        ],
        "max_tokens": 8192,
        "temperature": 0.6,
        "top_p": 0.95,
        "stream": False,
    }

    resultado = chamar_com_retry(OPENROUTER_URL, HEADERS, body)
    if resultado is None:
        raise RuntimeError(
            "Agente Planejador falhou após todas as tentativas — sem plano não há "
            "como continuar, abortando o pipeline."
        )
    return resultado


## Etapa 2 — Divide planejamento (parse + normalização do JSON retornado)

In [65]:

def sanear_json(texto):
    return re.sub(r',\s*([}\]])', r'\1', texto)


def divide_planejamento(resposta_api):
    """Equivalente ao node 'Divide planejamento': extrai e normaliza o(s) plano(s)
    retornado(s) pela LLM, aceitando os formatos { prontuarios: [...] }, [...] direto,
    ou um objeto único solto."""
    src = resposta_api
    raw = None

    if isinstance(src, dict) and src.get("choices"):
        raw = src["choices"][0]["message"]["content"]
    elif isinstance(src, dict) and src.get("body", {}).get("choices"):
        raw = src["body"]["choices"][0]["message"]["content"]
    elif isinstance(src, list) and src and src[0].get("choices"):
        raw = src[0]["choices"][0]["message"]["content"]
    else:
        raise ValueError(f"Divide planejamento: não encontrou 'content'. Chaves: {list(src.keys()) if isinstance(src, dict) else type(src)}")

    raw = raw.strip()
    if "```json" in raw:
        raw = raw.split("```json")[1].split("```")[0].strip()
    elif "```" in raw:
        raw = raw.split("```")[1].split("```")[0].strip()

    if raw.lower().startswith("json"):
        raw = raw[4:].strip()

    first_brace = raw.find("{")
    last_brace = raw.rfind("}")
    if first_brace != -1 and last_brace != -1 and first_brace < last_brace:
        raw = raw[first_brace:last_brace + 1]

    raw = sanear_json(raw)
    parsed = json.loads(raw)

    if isinstance(parsed, dict) and isinstance(parsed.get("prontuarios"), list):
        return parsed["prontuarios"]
    elif isinstance(parsed, list):
        return parsed
    elif isinstance(parsed, dict) and "id_planejamento" in parsed:
        return [parsed]
    else:
        raise ValueError(f"Divide planejamento: formato não reconhecido: {json.dumps(parsed)[:2000]}")


## Etapa 3 — Preenche dados pré-definidos (monta os registros de um plano)

In [ ]:

# ── CID faltante ─────────────────────────────────────────────────────────
# Quando a falha planejada é secao_c/diagnostico_cid = ausente, o CID tem que
# faltar de verdade: o campo "Cid procedimento" fica vazio E nenhum código CID
# pode aparecer nos campos abertos ("Descricao do registro"/"Descrição Cirurgica").
CID_ENTRE_PARENTESES = re.compile(r"\s*\((?:CID(?:-10)?\s*:?\s*)?[A-Za-z]\d{2}(?:\.\d{1,2})?\)", re.IGNORECASE)
CID_COM_ROTULO = re.compile(r",?\s*\bCID(?:-10)?\s*:?\s*[A-Za-z]\d{2}(?:\.\d{1,2})?", re.IGNORECASE)
CID_SOLTO_COM_DECIMAL = re.compile(r"\s*\b[A-Za-z]\d{2}\.\d{1,2}\b")


def cid_deve_faltar(falhas):
    """True se alguma falha planejada é 'CID cirúrgico ausente' (secao_c / diagnostico_cid)."""
    return any(
        f.get("secao") == "secao_c"
        and f.get("campo") == "diagnostico_cid"
        and (f.get("resultado") or "ausente") == "ausente"
        for f in (falhas or [])
    )


def remove_cids_do_texto(texto):
    """Tira códigos CID (ex: '(CID S72.1)', ', CID I50.0', 'S72.1') de um texto livre."""
    if not texto:
        return texto
    texto = CID_ENTRE_PARENTESES.sub("", texto)
    texto = CID_COM_ROTULO.sub("", texto)
    texto = CID_SOLTO_COM_DECIMAL.sub("", texto)
    return re.sub(r"[ \t]{2,}", " ", texto).replace(" .", ".").replace(" ,", ",")


def add_hours_to_date_string(data_str, horas):
    dt, time_part = _parse_data_hora(data_str)
    h, m = map(int, time_part.split(":"))
    dt = dt.replace(hour=h, minute=m) + timedelta(hours=horas)
    return f"{dt.day:02d}/{dt.month:02d}/{dt.year}, {dt.hour:02d}:{dt.minute:02d}"


def data_criacao_anamnese(plano, categoria):
    """Data de criação da anamnese. Se há falha planejada 'criacao_anamnese' para a
    anamnese desta categoria, ela é criada MAIS DE 12h após a internação (13h–20h)."""
    secao = "secao_b_anamnese" if categoria == "MEDICINA" else "secao_d_anamnese"
    atrasada = any(
        f.get("secao") == secao and f.get("campo") == "criacao_anamnese"
        for f in (plano.get("falhas_planejadas") or [])
    )
    if atrasada:
        return add_hours_to_date_string(plano["data_internacao"], random_int(13, 20))
    return plano["data_internacao"]


# ── Variabilidade temporal ───────────────────────────────────────────────
# Regras do auditor (prompt original 3.5 e 3.6): anamnese criada em até 12h da internação e uma evolução
# por janela de 24h até a alta (a janela do 1º dia é dispensada quando há anamnese). Sem sorteio, todos os
# prontuários cumpririam essas regras e o auditor nunca seria testado nelas. Por isso, ~30% dos
# prontuários recebem, em CÓDIGO (não pela LLM), UMA falha temporal sorteada.
PROPORCAO_FALHAS_TEMPORAIS = 0.30


def dias_com_evolucao_exigida(plano):
    """Dias (2..N) cuja evolução o auditor exige: N = janelas de 24h completas entre a internação e a alta."""
    dt_internacao = _parse_datahora_completa(plano["data_internacao"])
    dt_alta = _parse_datahora_completa(plano["data_saida"])
    janelas = int((dt_alta - dt_internacao).total_seconds() // 86400)
    return list(range(2, janelas + 1))


def _ja_tem_falha(plano, secao, campo):
    return any(f.get("secao") == secao and f.get("campo") == campo for f in (plano.get("falhas_planejadas") or []))


def adiciona_falhas_temporais(planos, proporcao=PROPORCAO_FALHAS_TEMPORAIS):
    """Sorteia ~`proporcao` dos planos para receber uma falha temporal: anamnese criada fora do prazo
    (médica ou de enfermagem) ou um dia sem evolução (médica ou de enfermagem). A falha entra em
    `falhas_planejadas` e, com isso, no gabarito; o plano vira PARCIALMENTE CONFORME."""
    for plano in planos:
        if random.random() >= proporcao:
            continue

        candidatas = []
        for secao, categoria, nome in (("secao_b_anamnese", "MEDICINA", "médica"), ("secao_d_anamnese", "ENFERMAGEM", "de enfermagem")):
            if not _ja_tem_falha(plano, secao, "criacao_anamnese"):
                candidatas.append({
                    "secao": secao, "campo": "criacao_anamnese", "resultado": "incompleta",
                    "categoria_profissional": categoria, "dias_ausentes": [],
                    "descricao_falha": f"Anamnese {nome} criada mais de 12 horas após a internação.",
                })

        try:
            dias = dias_com_evolucao_exigida(plano)
        except Exception:
            dias = []  # datas ausentes/malformadas: sem falha de evolução neste plano
        if dias:
            for secao, campo, categoria, nome in (("secao_b_evolucao", "frequencia_diaria", "MEDICINA", "médica"),
                                                  ("secao_d_evolucao", "criacao_evolucao", "ENFERMAGEM", "de enfermagem")):
                if not _ja_tem_falha(plano, secao, campo):
                    dia = random.choice(dias)
                    candidatas.append({
                        "secao": secao, "campo": campo, "resultado": "incompleta",
                        "categoria_profissional": categoria, "dias_ausentes": [dia],
                        "descricao_falha": f"Evolução {nome} ausente no {dia}º dia de internação (sem registro diário).",
                    })

        if not candidatas:
            continue
        plano.setdefault("falhas_planejadas", None)
        plano["falhas_planejadas"] = list(plano["falhas_planejadas"] or []) + [random.choice(candidatas)]
        plano["classificacao"] = "PARCIALMENTE CONFORME"
    return planos


def preenche_dados_predefinidos(plano):
    """Equivalente ao Code node 'Preenche dados pré-definidos'.
    Recebe UM plano (dict) e devolve a lista de registros (Anamnese/Evolução)
    daquele atendimento, cada um já com o _meta anexado."""
    p = plano
    prontuario = gerar_prontuario()
    atendimento = gerar_atendimento()
    codigo_sus = gerar_codigo_sus()
    nascimento = calcular_nascimento(p["idade_aproximada"], p["data_internacao"])

    falhas_por_categoria = {}
    for f in p.get("falhas_planejadas", []) or []:
        falhas_por_categoria.setdefault(f.get("categoria"), []).append(f)

    tem_cirurgia = p.get("tem_cirurgia", False)
    especialidade_cirurgia = p.get("especialidade", "") if tem_cirurgia else ""
    procedimento_cirurgico = p.get("procedimento", "") if tem_cirurgia else ""
    data_inicio_cirurgia = p.get("data_inicio_cirurgia", "") if tem_cirurgia else ""
    data_fim_cirurgia = p.get("data_fim_cirurgia", "") if tem_cirurgia else ""

    base = {
        "Prontuário": prontuario,
        "Atendimento": atendimento,
        "Data De Nascimento pact": nascimento,
        "Data da internação": p["data_internacao"],
        "Data de saída": p["data_saida"],
        "Data de óbito": "",
        "Sexo": p["sexo"],
        "Código Sus pact": codigo_sus,
        "Especialidade cirurgia": especialidade_cirurgia,
        "Procedimento cirurgico Realizado": procedimento_cirurgico,
        "Procedimento Interno Realizado": procedimento_cirurgico,
        "Cid procedimento": "" if cid_deve_faltar(p.get("falhas_planejadas")) else p.get("cid_principal", ""),
        "Data Inicio Cirurgia": data_inicio_cirurgia,
        "Data Fim Cirurgia": data_fim_cirurgia,
        "UF cirurgia": "BLOCO CIRURGICO" if tem_cirurgia else "",
        "Unidade Funcional Internaçao": p.get("unidade_funcional", ""),
        "Utilizou O2?": p.get("utiliza_o2", "Não"),
        "Usou Antibiótico Profilático?": p.get("antibiotico_profilatico", "Não"),
        "Seguiu protoc Cirurgia Segura?": "Não",
    }

    meta_unico = {
        "id_planejamento": p.get("id_planejamento"),
        "observacoes": p.get("observacoes_clinicas"),
        "falhas": p.get("falhas_planejadas", []) or [],
    }

    cirurgia_ja_marcada = False
    registros = []

    for categoria in p.get("categorias_envolvidas", []):
        falhas_categoria = falhas_por_categoria.get(categoria, [])
        dias_ausentes = [
            d
            for f in falhas_categoria
            for d in (f.get("dias_ausentes") or [])
        ]
        tem_falta_anamnese = any(
            f.get("tipo_registro") == "Anamnese" and "ausente" in (f.get("falha") or "").lower()
            for f in falhas_categoria
        )

        # Anamnese (dia 1)
        if categoria in ("MEDICINA", "ENFERMAGEM") and not tem_falta_anamnese:
            gerar_cirurgia = tem_cirurgia and not cirurgia_ja_marcada
            if gerar_cirurgia:
                cirurgia_ja_marcada = True

            registro = {
                **base,
                "Categoria Profissional": categoria,
                "Tipo do registro": "Anamnese",
                "criacao_anamnsese": data_criacao_anamnese(p, categoria),
                "Descricao do registro": f"[GERAR: Anamnese de {categoria} - {p['diagnostico_principal']}]",
                "Descrição Cirurgica": f"[GERAR: Descrição cirúrgica - {p.get('procedimento','')}]" if gerar_cirurgia else "",
            }
            registro["_meta"] = {**meta_unico, "gerar_cirurgia": True} if gerar_cirurgia else meta_unico
            registros.append(registro)

        # Evoluções (dia 2 em diante)
        for dia in range(2, p["duracao_internacao_dias"] + 1):
            if dia in dias_ausentes:
                continue

            gerar_cirurgia = tem_cirurgia and not cirurgia_ja_marcada
            if gerar_cirurgia:
                cirurgia_ja_marcada = True

            data_evolucao = add_days_to_date_string(p["data_internacao"], dia - 1)

            registro = {
                **base,
                "Categoria Profissional": categoria,
                "Tipo do registro": "Evolução",
                "criacao_anamnsese": data_evolucao,
                "Descricao do registro": f"[GERAR: Evolução dia {dia} de {categoria} - {p['diagnostico_principal']}]",
                "Descrição Cirurgica": f"[GERAR: Descrição cirúrgica - {p.get('procedimento','')}]" if gerar_cirurgia else "",
            }
            registro["_meta"] = {**meta_unico, "gerar_cirurgia": True} if gerar_cirurgia else meta_unico
            registros.append(registro)

    return registros


## Etapa 4 — Prepara body request (distribui falhas + monta prompt do Agente Gerador)

In [ ]:

SYSTEM_PROMPT_GERADOR = r"""# IDENTIDADE
Você é um agente GERADOR de texto clínico para prontuários eletrônicos hospitalares fictícios para treinamento de IA em auditoria de conformidade. Dados 100% simulados, sem correspondência real.

# ENTRADA
Um objeto JSON representando um registro de prontuário.
O objeto pode conter "_meta.falhas": uma lista de falhas planejadas. Se presente, APLIQUE rigorosamente estas falhas no texto gerado (ex: omita as informações solicitadas ou descreva de forma vaga). ATENÇÃO: Se a 'descricao_falha' mencionar um dia específico (ex: "no dia 2"), só aplique a falha se este registro atual for do dia correspondente (verifique a descrição e as datas). Caso contrário, ignore a falha e gere os dados normalmente.
Se a mensagem do usuário contiver uma seção "⏰ CHECAGEM TEMPORAL", ela foi CALCULADA a partir do horário real do registro comparado ao horário de início (e fim) da cirurgia. Não é obrigatório declarar isso explicitamente no texto (a data do registro já deixa isso implícito), mas SE o texto mencionar tempo de pós-operatório, número de PO, ou cirurgia já realizada, essa informação tem que bater EXATAMENTE com a CHECAGEM TEMPORAL — nunca com o que você inferir sozinho a partir do texto genérico "dia N" em "Descricao do registro" (esse "dia N" é só a ordem cronológica do atendimento, não indica se já houve cirurgia). Em caso de conflito, a CHECAGEM TEMPORAL sempre prevalece sobre qualquer suposição feita a partir do "dia N".

# REGRAS CRÍTICAS DE GERAÇÃO
0. PRIORIDADE MÁXIMA (FALHAS PLANEJADAS): Verifique a lista '_meta.falhas' no JSON. Se houver falhas, a regra ABSOLUTA é aplicá-las. Você é OBRIGADO a omitir ou tornar vagos os dados descritos na 'descricao_falha'. NUNCA inclua uma informação que a falha mandou tirar.
1. REALISMO CIRÚRGICO E TEXTUAL: Escreva o texto de forma contínua e natural. PROIBIDO começar o texto com títulos ou rótulos (ex: "ANAMNESE:", "EVOLUÇÃO MÉDICA:", "EVOLUÇÃO DE FISIOTERAPIA - DIA 3"). Comece o texto diretamente (ex: "Paciente no 3º PO de...").
2. SEM RÓTULOS ESTRUTURADOS: Não use marcações em formato de lista ou tópicos explícitos (ex: não escreva "HDA:", "AF:", "Exame Físico:"). A informação deve estar entrelaçada no texto corrido, simulando a escrita de um profissional com pressa.
3. CONFORMIDADE PADRÃO: A menos que a regra 0 (falhas planejadas) exija a omissão, você deve incluir o conteúdo a seguir de forma orgânica no texto:
   - ENFERMAGEM: Valor da Escala de Braden (informe o número, ex: Braden 15) e Escala de Morse (informe o número, ex: Morse 25). Na anamnese, descrever os antecedentes familiares.
   - MEDICINA: O conteúdo de HDA, Hipótese Diagnóstica/CID, Antecedentes Pessoais e Familiares, e Conduta.
4. EXAME FÍSICO COMPLETO: o auditor só aceita o exame se houver pelo menos UMA palavra de CADA sistema (a busca é por palavra inteira). Se não houver falha apontada para o exame físico, o texto DEVE cobrir TODOS os sistemas abaixo, usando os nomes ou termos clássicos indicados:
   - MEDICINA (5 sistemas): GERAL (ex: "bom estado geral"), ACV (ex: "acv: ritmo cardíaco regular, bulhas normofonéticas, sem sopros"), AR (ex: "ar: murmúrio vesicular presente, sem ruídos adventícios"), ABD (ex: "abdome flácido, indolor"), EXT (ex: "extremidades sem edemas, pulsos presentes").
   - ENFERMAGEM (8 itens): sistema nervoso (ex: "consciente, orientado"), pele/mucosas (ex: "pele corada, mucosas hidratadas"), respiratório (ex: "eupneico, murmúrio vesicular presente, saturação 96%"), cardiovascular (ex: "pulsos presentes, perfusão preservada"), gastrointestinal (ex: "abdome flácido, dieta aceita"), genitourinário (ex: "diurese presente"), músculo-esquelético (ex: "mobilidade preservada, deambulando com auxílio") e escalas (ex: "braden 15, morse 25").
   Se a falha planejada mandar omitir sistemas do exame, omita EXATAMENTE os citados (e nenhum outro) e não os mencione nem por sinônimos.
5. CURATIVO (somente ENFERMAGEM/Evolução): sempre que o texto citar curativo (realização, troca ou manutenção) e NÃO houver falha planejada para curativo, descreva: o TIPO ("curativo simples" para feridas simples, "curativo especial" para feridas complexas ou "curativo grau II" para lesões abertas extensas), o TAMANHO (ex: "tamanho aproximado 5 cm"), o EXSUDATO (ex: "sem exsudato") e a NECROSE (ex: "sem sinais de necrose"), usando exatamente as palavras curativo, tamanho, exsudato e necrose. Se houver falha planejada para curativo, omita o que ela mandar omitir (inclusive o tipo). Se não houver curativo a citar, não fale de curativo.

# EXEMPLO DE ESTILO ESPERADO (SUJO E REALISTA)
NÃO ESCREVA ASSIM: "Paciente encontra-se no 2º dia pós-operatório. Ao exame físico, apresenta..."
ESCREVA ASSIM: "paciente no 2po de colelap, evolui bem, sem queixas no momento. aceitou dieta. abdome flacido, indolor, ferida operatoria seca e sem sinais flogisticos. extremidades aquecidas. braden 15, morse risco moderado. segue conduta da cirurgia."

# SAÍDA
Sua única tarefa é gerar o texto clínico. Você DEVE retornar EXATAMENTE este formato JSON, contendo APENAS estas duas chaves. NÃO repita os outros dados do paciente:

{
  "Descricao do registro": "Seu texto clínico gerado aqui, seguindo as regras de conformidade e o estilo do exemplo...",
  "Descrição Cirurgica": "Sua descrição cirúrgica crua e realista aqui (SE EXIGIDO) ou uma string vazia \"\" (SE NÃO EXIGIDO)"
}

Sem explicações, sem markdown, sem blocos de código. Apenas o JSON puro {}."""


MAPA_SECAO = {
    "secao_b_anamnese": {"categoria": "MEDICINA", "tipo": "Anamnese"},
    "secao_b_evolucao": {"categoria": "MEDICINA", "tipo": "Evolução"},
    "secao_d_anamnese": {"categoria": "ENFERMAGEM", "tipo": "Anamnese"},
    "secao_d_evolucao": {"categoria": "ENFERMAGEM", "tipo": "Evolução"},
    "secao_c": {"categoria": "MEDICINA", "tipo": None},
}


def extrair_dia(descricao_placeholder):
    m = re.search(r"dia\s+(\d+)", descricao_placeholder or "", re.IGNORECASE)
    return int(m.group(1)) if m else 1


def _parse_datahora_completa(data_str):
    """'DD/MM/AAAA, HH:MM' -> datetime completo (data + hora), para permitir
    comparar HORÁRIOS de verdade (não só datas) entre o registro e a cirurgia."""
    dt, hora = _parse_data_hora(data_str)
    h, m = map(int, hora.split(":"))
    return dt.replace(hour=h, minute=m)


def checagem_temporal_cirurgia(registro):
    """Compara explicitamente o horário do PRÓPRIO registro com o horário de
    início (e fim) da cirurgia do atendimento, e devolve um bloco de texto
    dizendo à LLM, sem ambiguidade, se aquele registro é PRÉ-OPERATÓRIO,
    TRANSOPERATÓRIO ou PÓS-OPERATÓRIO — e, se pós-operatório, qual o número
    de PO correto (calculado, não inferido pela LLM a partir do texto
    genérico "dia N" de 'Descricao do registro').

    Devolve string vazia se o atendimento não tem cirurgia ou se alguma das
    datas necessárias estiver ausente/malformada (nesse caso a checagem
    simplesmente não é anexada, sem travar a geração)."""
    data_inicio_cirurgia = registro.get("Data Inicio Cirurgia") or ""
    if not data_inicio_cirurgia:
        return ""  # atendimento sem cirurgia — checagem não se aplica

    data_registro_str = registro.get("criacao_anamnsese") or ""
    if not data_registro_str:
        return ""

    try:
        dt_registro = _parse_datahora_completa(data_registro_str)
        dt_inicio_cir = _parse_datahora_completa(data_inicio_cirurgia)
    except Exception:
        return ""  # datas malformadas — não bloqueia a geração, só não anota

    dt_fim_cir = None
    data_fim_cirurgia = registro.get("Data Fim Cirurgia") or ""
    if data_fim_cirurgia:
        try:
            dt_fim_cir = _parse_datahora_completa(data_fim_cirurgia)
        except Exception:
            dt_fim_cir = None

    dias_apos_cirurgia = (dt_registro.date() - dt_inicio_cir.date()).days
    fmt = lambda dt: dt.strftime("%d/%m/%Y %H:%M")

    aviso_opcional = (
        "Não é obrigatório declarar isso explicitamente no texto (a data do registro já deixa "
        "isso implícito) — mas SE o texto mencionar tempo de pós-operatório, cirurgia já feita, "
        "ou algo do tipo, a informação tem que bater com o que está calculado abaixo, sem exceção."
    )

    if dt_fim_cir and dt_inicio_cir <= dt_registro <= dt_fim_cir:
        status = (
            f"O horário deste registro ({fmt(dt_registro)}) cai DENTRO do intervalo em que a "
            f"cirurgia estava ocorrendo ({fmt(dt_inicio_cir)} até {fmt(dt_fim_cir)}). "
            f"Trate como TRANSOPERATÓRIO / imediatamente pós-cirúrgico SE mencionar isso — NÃO "
            f"descreva um PO avançado nem uma recuperação já em curso. {aviso_opcional}"
        )
    elif dt_registro < dt_inicio_cir:
        status = (
            f"O horário deste registro ({fmt(dt_registro)}) é ANTERIOR ao horário de início da "
            f"cirurgia ({fmt(dt_inicio_cir)}). Logo, este registro é PRÉ-OPERATÓRIO — o paciente "
            f"AINDA NÃO FOI OPERADO. SE mencionar a cirurgia, NÃO escreva 'Nº PO', 'pós-operatório' "
            f"nem qualquer coisa que sugira que ela já foi realizada (o quadro pré-cirúrgico cabível "
            f"seria preparo, jejum, avaliação pré-anestésica, ansiedade pré-operatória, etc.). "
            f"{aviso_opcional}"
        )
    elif dias_apos_cirurgia == 0:
        status = (
            f"O horário deste registro ({fmt(dt_registro)}) é POSTERIOR ao horário de início da "
            f"cirurgia ({fmt(dt_inicio_cir)}), mas ainda no MESMO DIA em que ela ocorreu (0 dias "
            f"corridos de diferença). Logo, este registro é PÓS-OPERATÓRIO IMEDIATO — SE mencionar "
            f"isso, use a expressão 'PO imediato' (SEM número, ex: 'evolui bem no pós-operatório "
            f"imediato'). NUNCA escreva '0º PO' nem qualquer número de PO para este registro. "
            f"{aviso_opcional}"
        )
    else:
        status = (
            f"O horário deste registro ({fmt(dt_registro)}) é POSTERIOR ao horário de início da "
            f"cirurgia ({fmt(dt_inicio_cir)}), {dias_apos_cirurgia} dia(s) corridos depois dela "
            f"(dias diferentes no calendário). Logo, este registro é PÓS-OPERATÓRIO — SE mencionar "
            f"o número do PO, use EXATAMENTE '{dias_apos_cirurgia}º PO' (ou variação natural "
            f"equivalente). NUNCA use um número de PO diferente deste, mesmo que o texto de "
            f"referência 'Descricao do registro' sugira outro dia. {aviso_opcional}"
        )

    return (
        "\n\n⏰ CHECAGEM TEMPORAL (calculada a partir das datas reais, para você não errar SE for "
        "mencionar isso — NÃO infira sozinho a partir do texto placeholder):\n"
        f"{status}"
    )


def falha_se_aplica_ao_registro(f, registro, dia_do_registro):
    categoria = registro["Categoria Profissional"]
    tipo = registro["Tipo do registro"]

    if f.get("secao") == "secao_e":
        if categoria in ("MEDICINA", "ENFERMAGEM"):
            return False
        if f.get("categoria_profissional") and f["categoria_profissional"] != categoria:
            return False
    else:
        esperado = MAPA_SECAO.get(f.get("secao"))
        if not esperado:
            return False
        if categoria != esperado["categoria"]:
            return False
        if esperado["tipo"] is not None and tipo != esperado["tipo"]:
            return False

    dias_ausentes = f.get("dias_ausentes") or []
    if dias_ausentes:
        return dia_do_registro in dias_ausentes
    return True


def prepara_body_request(registro):
    """Equivalente ao Code node 'Prepara body request'. Recebe UM registro (já
    com _meta.falhas do atendimento inteiro) e devolve o dict pronto para a
    chamada HTTP ao Agente Gerador — ou {'tipo_geracao': 'descartar', ...}
    se o registro não deveria ser gerado (evolução ausente naquele dia)."""
    tem_cirurgia = registro["Especialidade cirurgia"] != ""
    eh_primeiro_cirurgico = (registro.get("_meta") or {}).get("gerar_cirurgia") is True

    falhas_do_atendimento = (registro.get("_meta") or {}).get("falhas") or []
    dia_do_registro = extrair_dia(registro["Descricao do registro"])

    falhas_deste_registro = [
        f for f in falhas_do_atendimento
        if falha_se_aplica_ao_registro(f, registro, dia_do_registro)
    ]

    deve_ser_descartado = any(
        f.get("campo") in ("frequencia_diaria", "criacao_evolucao")
        and dia_do_registro in (f.get("dias_ausentes") or [])
        for f in falhas_deste_registro
    )
    if deve_ser_descartado:
        return {
            "tipo_geracao": "descartar",
            "prontuario_id": registro["Prontuário"],
            "motivo": "evolução ausente neste dia",
        }

    registro_sem_meta = {k: v for k, v in registro.items() if k != "_meta"}

    system_prompt = SYSTEM_PROMPT_GERADOR
    if falhas_deste_registro:
        bullets = "\n".join(f"-> {f.get('descricao_falha','')}" for f in falhas_deste_registro)
        system_prompt += (
            "\n\n🚨 ATENÇÃO MÁXIMA PARA ESTE REGISTRO! VOCÊ É OBRIGADO A APLICAR AS "
            f"SEGUINTES FALHAS NESTE TEXTO:\n{bullets}"
        )

    if cid_deve_faltar(falhas_do_atendimento):
        system_prompt += (
            "\n\n🚫 CID AUSENTE NESTE ATENDIMENTO: NÃO escreva NENHUM código CID (ex: 'S72.1', 'I50.0', "
            "'(CID ...)') em NENHUM campo — nem em 'Descricao do registro' nem em 'Descrição Cirurgica'. "
            "Cite o diagnóstico apenas por extenso."
        )

    instrucao_cirurgia = (
        "⚠️ INSTRUÇÃO CRÍTICA: Este é o primeiro registro cirúrgico. Você DEVE gerar "
        "a 'Descrição Cirurgica' detalhada neste JSON."
        if eh_primeiro_cirurgico else
        "⚠️ INSTRUÇÃO: NÃO gere a 'Descrição Cirurgica' neste JSON. Deixe o campo como string vazia \"\"."
    )

    instrucao_temporal = checagem_temporal_cirurgia(registro)

    user_content = (
        f"DADOS DO REGISTRO:\n{json.dumps(registro_sem_meta, indent=2, ensure_ascii=False)}\n\n"
        f"{instrucao_cirurgia}"
        f"{instrucao_temporal}"
    )

    body = {
        "model": MODELO_GERADOR,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
        "max_tokens": 4096,
        "temperature": 0.60,
        "top_p": 0.95,
        "stream": False,
    }

    return {
        "tipo_geracao": "registro",
        "prontuario_id": registro["Prontuário"],
        "body": body,
        "_registro_original": registro,  # mantido em memória p/ o merge (substitui o Merge1 do n8n)
        "_falhas_deste_registro": falhas_deste_registro,  # para conferir se o texto gerado respeita as regras
    }


## Etapa 5 — Chamada ao Agente Gerador + Etapa 6 — Monta json final

In [ ]:

def chamar_agente_gerador(body):
    """Retorna o JSON da resposta, ou None se esgotar as tentativas de retry --
    nesse caso o registro correspondente deve ser descartado pelo chamador.
    Se o servidor sinalizar sobrecarga (429/5xx), cai automaticamente para
    MODELO_GERADOR_FALLBACK (ver Etapa 4/Configuração) só nesse caso -- para
    qualquer outro erro, continua tentando com o modelo original (flash) e
    o backoff normal de 5s/10s/20s..."""
    return chamar_com_retry(OPENROUTER_URL, HEADERS, body, modelo_fallback=MODELO_GERADOR_FALLBACK)


def monta_json_final(registro_original, resposta_api):
    """Equivalente ao Code node 'Monta json final': extrai o conteúdo gerado
    pela LLM e mescla no registro original (o merge por posição do n8n vira,
    aqui, simplesmente juntar os dois objetos em memória)."""
    registro_final = dict(registro_original)

    conteudo = ""
    if resposta_api.get("choices"):
        conteudo = resposta_api["choices"][0]["message"]["content"]
    elif resposta_api.get("content"):
        conteudo = resposta_api["content"]

    if not conteudo:
        registro_final["_erro_parse"] = "resposta da API sem 'choices'/'content'"
        return registro_final

    texto = conteudo.strip()
    texto = texto.replace("```json", "").replace("```", "").strip()

    if texto.lower().startswith("json"):
        texto = texto[4:].strip()

    if texto.startswith('"Descricao'):
        texto = "{\n" + texto

    if not texto.endswith("}"):
        texto = texto + "\n}"

    texto = sanear_json(texto)

    try:
        json_llm = json.loads(texto, strict=False)  # strict=False: aceita quebras de linha cruas dentro das strings
        if json_llm.get("Descricao do registro"):
            registro_final["Descricao do registro"] = json_llm["Descricao do registro"]
        desc_cir = json_llm.get("Descrição Cirurgica", "")
        if desc_cir and desc_cir.strip() and "[GERAR:" not in desc_cir:
            registro_final["Descrição Cirurgica"] = desc_cir
    except json.JSONDecodeError as e:
        registro_final["_erro_parse"] = str(e)
        registro_final["_conteudo_bruto_llm"] = texto
        return registro_final

    # CID faltante: garante que nenhum código CID sobrou nos campos abertos
    if cid_deve_faltar((registro_original.get("_meta") or {}).get("falhas")):
        for campo in ("Descricao do registro", "Descrição Cirurgica"):
            registro_final[campo] = remove_cids_do_texto(registro_final.get(campo, ""))

    return registro_final


# ── Regras do auditor reaproveitadas na validação do texto gerado ─────────
# O texto gerado só é gravado se respeitar as regras que o auditor vai aplicar: exame físico completo
# e curativo descrito por inteiro quando NÃO há falha planejada, e a falha realmente aplicada quando HÁ.
import sys
from pathlib import Path


def _raiz_do_repositorio():
    for pasta in [Path.cwd(), *Path.cwd().parents]:
        if (pasta / "data_extract" / "core" / "auditor.py").exists():
            return pasta
    raise RuntimeError("Não encontrei a pasta data_extract acima de " + str(Path.cwd()))


_raiz = _raiz_do_repositorio()
if str(_raiz) not in sys.path:
    sys.path.insert(0, str(_raiz))

from data_extract.utils.helpers import check_exame_fisico_completo, check_curativo
from data_extract.keywords.exame_fisico import TERMOS_EXAME_FISICO_MEDICINA, TERMOS_EXAME_FISICO_ENFERMAGEM


def registro_respeita_regras(registro, falhas_do_registro):
    """None se o texto respeita as regras do auditor; senão, o motivo (usado para gerar de novo)."""
    categoria, tipo = registro.get("Categoria Profissional"), registro.get("Tipo do registro")
    texto = registro.get("Descricao do registro") or ""
    planejadas = {f.get("campo") for f in (falhas_do_registro or [])}

    dicionario = {"MEDICINA": TERMOS_EXAME_FISICO_MEDICINA, "ENFERMAGEM": TERMOS_EXAME_FISICO_ENFERMAGEM}.get(categoria)
    if dicionario is not None:
        exame = check_exame_fisico_completo(texto, dicionario)
        if "exame_fisico" in planejadas and exame == "conforme":
            return "a falha planejada de exame físico não foi aplicada: o exame veio completo"
        if "exame_fisico" not in planejadas and exame != "conforme":
            return f"exame físico incompleto sem falha planejada ({exame}); inclua TODOS os sistemas"

    if categoria == "ENFERMAGEM" and tipo == "Evolução":
        curativo = check_curativo(texto)
        if curativo != "Não se aplica":
            if "curativo" in planejadas and curativo.startswith("conforme"):
                return "a falha planejada de curativo não foi aplicada: o curativo veio completo"
            if "curativo" not in planejadas and not curativo.startswith("conforme"):
                return ("curativo citado sem descrição completa; descreva o tipo (simples, especial ou grau II), "
                        "o tamanho, o exsudato e a necrose, ou não cite curativo")
    return None


def registro_gerado_ok(registro_final, precisa_desc_cirurgica):
    """Devolve None se o registro veio CORRETO, ou o motivo (str) se não deve ser gravado:
    erro de parse, texto vazio ou com a tag [GERAR (a LLM não substituiu o placeholder),
    ou descrição cirúrgica que deveria ter vindo e não veio."""
    if registro_final.get("_erro_parse"):
        return f"erro ao ler a resposta da LLM ({registro_final['_erro_parse']})"
    desc = registro_final.get("Descricao do registro") or ""
    if not desc.strip():
        return "descrição do registro vazia"
    if "[GERAR" in desc:
        return "descrição do registro ainda com a tag [GERAR"
    if precisa_desc_cirurgica:
        cir = registro_final.get("Descrição Cirurgica") or ""
        if not cir.strip():
            return "descrição cirúrgica não veio na resposta"
        if "[GERAR" in cir:
            return "descrição cirúrgica ainda com a tag [GERAR"
    return None


## Etapa 6b — Uniformiza a Descrição Cirurgica dentro do mesmo atendimento

Só UM registro do atendimento é instruído a gerar a `Descrição Cirurgica` de
verdade (o marcado com `gerar_cirurgia=True` em `prepara_body_request`) — os
demais recebem instrução para deixar o campo vazio. Aqui, depois que todos os
registros do atendimento já foram gerados, copiamos essa descrição para TODOS
os registros que tenham cirurgia (`Especialidade cirurgia` preenchida), para
que ela seja idêntica em todo o prontuário — sem precisar de uma nova
requisição por registro.

In [69]:

def uniformizar_descricao_cirurgica(registros_do_atendimento):
    """Garante que todos os registros do MESMO atendimento com cirurgia
    fiquem com a MESMA 'Descrição Cirurgica' — a que foi de fato gerada pela
    LLM em um único registro, replicada para os demais."""
    descricao_cirurgia = ""
    for reg in registros_do_atendimento:
        desc = reg.get("Descrição Cirurgica", "")
        if desc and desc.strip() and "[GERAR:" not in desc:
            descricao_cirurgia = desc
            break

    if not descricao_cirurgia:
        return registros_do_atendimento

    for reg in registros_do_atendimento:
        if reg.get("Especialidade cirurgia"):
            reg["Descrição Cirurgica"] = descricao_cirurgia

    return registros_do_atendimento


## Etapa 7 — Prepara json e csv (agrega falhas por atendimento + monta gabarito)

In [70]:

SECAO_INFO = {
    "secao_b_anamnese": {"categoria": "MEDICINA", "tipo": "Anamnese"},
    "secao_b_evolucao": {"categoria": "MEDICINA", "tipo": "Evolução"},
    "secao_d_anamnese": {"categoria": "ENFERMAGEM", "tipo": "Anamnese"},
    "secao_d_evolucao": {"categoria": "ENFERMAGEM", "tipo": "Evolução"},
    "secao_c": {"categoria": "MEDICINA", "tipo": "Cirúrgico"},
}


def _csv_escape(valor):
    return (valor or "").replace('"', '""')


def prepara_json_e_csv(registros_finais, incluir_cabecalho=True):
    """Equivalente ao Code node 'Prepara json e csv'.
    Recebe a lista de registros já finalizados (todos os atendimentos do lote)
    e devolve (linhas_csv, linhas_txt, lista_de_prontuarios_processados)."""

    # ── Agrega falhas de todos os registros do mesmo Atendimento ──────────
    falhas_por_atendimento = {}
    falhas_vistas = {}
    for reg in registros_finais:
        atendimento = reg.get("Atendimento")
        falhas = (reg.get("_meta") or {}).get("falhas") or []
        falhas_por_atendimento.setdefault(atendimento, [])
        falhas_vistas.setdefault(atendimento, set())
        for f in falhas:
            chave = f"{f.get('secao')}|{f.get('campo')}|{f.get('categoria_profissional','')}"
            if chave in falhas_vistas[atendimento]:
                continue
            falhas_vistas[atendimento].add(chave)
            falhas_por_atendimento[atendimento].append(f)

    csv_header = '"record","section","field","professional category","record type","result","description missing","total conformity"'
    csv_linhas = [csv_header] if incluir_cabecalho else []
    txt_linhas = []
    prontuarios_processados = set()

    for reg in registros_finais:
        atendimento = reg.get("Atendimento")
        falhas = falhas_por_atendimento.get(atendimento, [])
        num = _csv_escape(reg.get("Prontuário"))

        # nunca grava chaves internas (_meta, _erro_parse, _conteudo_bruto_llm...)
        reg_sem_meta = {k: v for k, v in reg.items() if not k.startswith("_")}
        txt_linhas.append(json.dumps(reg_sem_meta, indent=2, ensure_ascii=False))

        if num in prontuarios_processados:
            continue
        prontuarios_processados.add(num)

        conformidade = "Parcialmente Conforme" if falhas else "Conforme"

        if not falhas:
            csv_linhas.append(f'"{num}","","","","","","","{conformidade}"')
        else:
            for f in falhas:
                info = SECAO_INFO.get(f.get("secao"), {})
                secao = _csv_escape(f.get("secao"))
                campo = _csv_escape(f.get("campo"))
                cat = _csv_escape(f.get("categoria_profissional") or info.get("categoria", ""))
                tipo_f = _csv_escape(info.get("tipo", ""))
                resultado = _csv_escape(f.get("resultado"))
                descricao = _csv_escape(f.get("descricao_falha"))
                csv_linhas.append(
                    f'"{num}","{secao}","{campo}","{cat}","{tipo_f}","{resultado}","{descricao}","{conformidade}"'
                )

    return csv_linhas, txt_linhas, list(prontuarios_processados)


## Etapa 8 — Orquestração completa (equivalente ao "Loop Over Items" + salvar arquivos)

In [ ]:

def _carregar_json_existente(caminho):
    """Lê o array JSON já salvo (execuções anteriores) e devolve a lista de registros.

    - Arquivo inexistente ou vazio -> [].
    - Arquivo inválido (ex: execução anterior interrompida no meio): NÃO perde nada.
      Guarda uma cópia do original como '<arquivo>.corrompido-<data>' e recupera todos
      os registros completos que conseguir ler, descartando só o pedaço truncado."""
    if not os.path.exists(caminho):
        return []

    with open(caminho, "r", encoding="utf-8") as f:
        conteudo = f.read()

    if not conteudo.strip():
        return []

    try:
        dados = json.loads(conteudo)
        if isinstance(dados, list):
            return dados
        raise ValueError("o JSON existente não é um array")
    except ValueError as erro:
        backup = f"{caminho}.corrompido-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
        with open(backup, "w", encoding="utf-8") as bf:
            bf.write(conteudo)
        print(f"AVISO: {caminho} estava inválido ({erro}). Cópia salva em {backup}; recuperando os registros completos...")

    recuperados = []
    decoder = json.JSONDecoder()
    i = conteudo.find("[") + 1
    while i > 0 and i < len(conteudo):
        while i < len(conteudo) and conteudo[i] in " \n\r\t,":
            i += 1
        if i >= len(conteudo) or conteudo[i] == "]":
            break
        try:
            item, i = decoder.raw_decode(conteudo, i)
        except ValueError:
            break  # resto truncado/corrompido
        if isinstance(item, dict):
            recuperados.append(item)
    print(f"   -> {len(recuperados)} registro(s) recuperado(s).")
    return recuperados


def _gravar_json_atomico(caminho, registros):
    """Grava o array COMPLETO em um arquivo temporário e troca pelo definitivo com
    os.replace (atômico). Assim o arquivo em disco é SEMPRE um array JSON válido:
    se o kernel cair, o Jupyter for interrompido ou a API falhar no meio, o que fica
    é a última versão completa — nunca um arquivo pela metade."""
    texto = json.dumps(registros, indent=2, ensure_ascii=False) + "\n"
    json.loads(texto)  # garantia extra: só troca o arquivo se o conteúdo é JSON válido

    tmp = caminho + ".tmp"
    try:
        with open(tmp, "w", encoding="utf-8") as f:
            f.write(texto)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, caminho)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


MAX_REGENERACOES = 4  # quantas vezes tenta gerar de novo um registro que veio inválido


def gerar_registro_valido(preparado, max_tentativas=MAX_REGENERACOES):
    """Chama o Agente Gerador até vir um registro CORRETO: sem erro de parse, sem a tag [GERAR, com a
    descrição cirúrgica quando exigida e respeitando as regras do auditor (exame físico completo, curativo
    completo — ou a falha planejada de fato aplicada). Em cada nova tentativa, o motivo da rejeição anterior
    vai junto no pedido. Devolve (registro, None) em caso de sucesso, ou (None, motivo) se esgotar as tentativas."""
    original = preparado["_registro_original"]
    falhas = preparado.get("_falhas_deste_registro") or []
    precisa_cirurgia = bool((original.get("_meta") or {}).get("gerar_cirurgia"))
    body = json.loads(json.dumps(preparado["body"]))  # cópia: o aviso de correção não vaza para outros registros
    motivo = "sem resposta da API"

    for tentativa in range(1, max_tentativas + 1):
        resposta = chamar_agente_gerador(body)
        if resposta is None:
            motivo = "sem resposta da API após todas as tentativas de rede"
        else:
            registro_final = monta_json_final(original, resposta)
            motivo = registro_gerado_ok(registro_final, precisa_cirurgia)
            if motivo is None:
                motivo = registro_respeita_regras(registro_final, falhas)
            if motivo is None:
                return registro_final, None
        print(f"   -> registro inválido ({motivo}) — tentativa {tentativa}/{max_tentativas}")
        body["messages"][-1]["content"] = (
            preparado["body"]["messages"][-1]["content"]
            + f"\n\n⚠️ CORREÇÃO OBRIGATÓRIA: a tentativa anterior foi rejeitada porque: {motivo}. Gere de novo corrigindo isso."
        )

    return None, motivo


def rodar_pipeline(quantidade_prontuarios=QUANTIDADE_PRONTUARIOS, primeira_execucao=True):
    """Roda o pipeline completo, salvando CADA PRONTUÁRIO no disco assim que ele
    termina de ser gerado — não espera todos os N prontuários ficarem prontos.
    Assim, se algo falhar no meio (ex: prontuário 3 de 5), o que já foi gerado
    dos prontuários 1 e 2 já está salvo em disco.

    O JSON_PATH é SEMPRE um array JSON válido ([ {...}, {...} ]), em qualquer momento:
    a cada prontuário o array completo é regravado de forma atômica (arquivo temporário
    + os.replace), então não existe estado intermediário inválido, nem em caso de erro,
    interrupção do kernel ou queda de energia."""
    print(f"1) Chamando Agente Planejador para {quantidade_prontuarios} prontuário(s)...")
    resposta_planejador = chamar_agente_planejador(quantidade_prontuarios)

    print("2) Dividindo o planejamento em planos individuais...")
    planos = divide_planejamento(resposta_planejador)
    print(f"   -> {len(planos)} plano(s) recebido(s)")

    antes = sum(1 for pl in planos if pl.get("falhas_planejadas"))
    planos = adiciona_falhas_temporais(planos)
    depois = sum(1 for pl in planos if pl.get("falhas_planejadas"))
    print(f"   -> falhas temporais sorteadas (~{PROPORCAO_FALHAS_TEMPORAIS:.0%}): {depois - antes} plano(s) receberam falha de prazo/frequência")

    modo_csv = "w" if primeira_execucao else "a"
    cabecalho_pendente = primeira_execucao  # só inclui cabeçalho no 1º prontuário salvo

    total_prontuarios_salvos = 0
    total_registros_salvos = 0
    todos_registros_finais = []  # devolvido no final, útil para inspecionar na própria sessão

    arquivo_csv = open(CSV_PATH, modo_csv, encoding="utf-8")

    # Na primeira execução começa um array novo; nas seguintes continua o que já existe.
    registros_json = [] if primeira_execucao else _carregar_json_existente(JSON_PATH)
    _gravar_json_atomico(JSON_PATH, registros_json)  # já deixa o arquivo válido desde o início

    try:
        for i, plano in enumerate(planos, start=1):
            print(f"\n3) Processando plano {i}/{len(planos)} (id_planejamento={plano.get('id_planejamento')})...")

            registros = preenche_dados_predefinidos(plano)
            print(f"   -> {len(registros)} registro(s) planejado(s) para este atendimento")

            registros_finais_deste_plano = []
            plano_abortado = None

            for j, reg in enumerate(registros, start=1):
                preparado = prepara_body_request(reg)

                if preparado.get("tipo_geracao") == "descartar":
                    print(f"   -> descartado: {preparado['motivo']} (prontuario {preparado['prontuario_id']})")
                    continue

                print(f"   Gerando prontuário {i} - registro {j}/{len(registros)}...")
                registro_final, motivo = gerar_registro_valido(preparado)

                if registro_final is None:
                    # Um registro faltando deixaria o prontuário incoerente com o gabarito:
                    # em vez de gravar pela metade, o prontuário INTEIRO é descartado.
                    plano_abortado = f"registro {j}/{len(registros)}: {motivo}"
                    break

                registros_finais_deste_plano.append(registro_final)

            if plano_abortado:
                print(f"   -> plano {i} ({plano.get('id_planejamento')}) DESCARTADO, nada foi gravado. Motivo: {plano_abortado}")
                continue

            if not registros_finais_deste_plano:
                print(f"   -> nenhum registro válido neste plano, nada a salvar.")
                continue

            registros_finais_deste_plano = uniformizar_descricao_cirurgica(registros_finais_deste_plano)

            # Última barreira antes de gravar: nenhum registro pode ter tag [GERAR nem erro de parse
            problemas = [
                registro_gerado_ok(r, precisa_desc_cirurgica=bool(r.get("Especialidade cirurgia")))
                for r in registros_finais_deste_plano
            ]
            problemas = [pb for pb in problemas if pb]
            if problemas:
                print(f"   -> plano {i} ({plano.get('id_planejamento')}) DESCARTADO, nada foi gravado. Motivo: {problemas[0]}")
                continue

            # ── Salva ESTE prontuário imediatamente, sem esperar os outros ──
            csv_linhas, txt_linhas, numeros_prontuario = prepara_json_e_csv(
                registros_finais_deste_plano, incluir_cabecalho=cabecalho_pendente
            )
            cabecalho_pendente = False  # só o primeiro salvamento leva cabeçalho

            arquivo_csv.write("\n".join(csv_linhas) + "\n")
            arquivo_csv.flush()

            registros_json.extend(json.loads(linha) for linha in txt_linhas)
            _gravar_json_atomico(JSON_PATH, registros_json)

            total_prontuarios_salvos += len(numeros_prontuario)
            total_registros_salvos += len(registros_finais_deste_plano)
            todos_registros_finais.extend(registros_finais_deste_plano)

            for num in numeros_prontuario:
                print(f"Prontuário {num} gerado com sucesso e salvo no gabarito.csv e prontuarios.json")

    finally:
        arquivo_csv.close()  # o JSON não precisa de fechamento: já está válido em disco

    print(f"\nConcluído: {total_prontuarios_salvos} prontuário(s), "
          f"{total_registros_salvos} registro(s) salvos no total.")
    print(f"CSV: {CSV_PATH}")
    print(f"JSON: {JSON_PATH}")

    return todos_registros_finais


## Executar

In [72]:

# Ajuste a quantidade e rode. Para gerar mais lotes depois, chame de novo com
# primeira_execucao=False (equivale a rodar o "Loop Over Items" mais uma vez
# no n8n, fazendo append nos arquivos existentes).

registros_gerados = rodar_pipeline(quantidade_prontuarios=3, primeira_execucao=False)


1) Chamando Agente Planejador para 3 prontuário(s)...
2) Dividindo o planejamento em planos individuais...
   -> 3 plano(s) recebido(s)

3) Processando plano 1/3 (id_planejamento=PLAN-001)...
   -> 6 registro(s) planejado(s) para este atendimento
   Gerando prontuário 1 - registro 1/6...
   -> registro inválido (descrição do registro ainda com a tag [GERAR) — tentativa 1/3
   -> registro inválido (erro ao ler a resposta da LLM (Expecting value: line 2 column 1 (char 1))) — tentativa 2/3
   Gerando prontuário 1 - registro 2/6...
   Gerando prontuário 1 - registro 3/6...
   Gerando prontuário 1 - registro 4/6...
   Gerando prontuário 1 - registro 5/6...
   Gerando prontuário 1 - registro 6/6...
Prontuário 26.696.588 gerado com sucesso e salvo no gabarito.csv e prontuarios.json

3) Processando plano 2/3 (id_planejamento=PLAN-002)...
   -> 6 registro(s) planejado(s) para este atendimento
   Gerando prontuário 2 - registro 1/6...
   -> descartado: evolução ausente neste dia (prontuario 55.3